Project For CNS

In [1]:
# ============================================================
# SECURE E-MAIL COMMUNICATION USING RSA AND AES
# ============================================================

# Install PyCryptodome before running:
# pip install pycryptodome

# ============================================================
# IMPORT REQUIRED LIBRARIES
# ============================================================

import os
import json
import base64

from Crypto.Cipher import AES, PKCS1_OAEP
from Crypto.PublicKey import RSA
from Crypto.Random import get_random_bytes
from Crypto.Signature import pkcs1_15
from Crypto.Hash import SHA256


# ============================================================
# DISPLAY PROJECT TITLE
# ============================================================

print("=" * 60)
print("SECURE E-MAIL COMMUNICATION USING RSA AND AES")
print("=" * 60)


# ============================================================
# CREATE REQUIRED FOLDERS
# ============================================================

os.makedirs("Keys", exist_ok=True)
os.makedirs("Messages", exist_ok=True)

print("Folders created successfully.")


# ============================================================
# FUNCTION TO GENERATE RSA KEYS
# ============================================================

def generate_keys(user_name):

    # Generate RSA 2048-bit key pair
    key = RSA.generate(2048)

    # Export private and public keys
    private_key = key.export_key()
    public_key = key.publickey().export_key()

    # Save private key
    with open(f"Keys/{user_name}_Private.pem", "wb") as f:
        f.write(private_key)

    # Save public key
    with open(f"Keys/{user_name}_Public.pem", "wb") as f:
        f.write(public_key)

    print(f"{user_name} RSA keys generated.")


# ============================================================
# GENERATE KEYS FOR SENDER AND RECEIVER
# ============================================================

generate_keys("Sender")
generate_keys("Receiver")


# ============================================================
# FUNCTION TO LOAD PUBLIC KEY
# ============================================================

def load_public_key(path):

    with open(path, "rb") as f:
        key = RSA.import_key(f.read())

    return key


# ============================================================
# FUNCTION TO LOAD PRIVATE KEY
# ============================================================

def load_private_key(path):

    with open(path, "rb") as f:
        key = RSA.import_key(f.read())

    return key


# ============================================================
# AES ENCRYPTION FUNCTION
# ============================================================

def aes_encrypt(message):

    # Generate random 256-bit AES key
    aes_key = get_random_bytes(32)

    # Create AES cipher in EAX mode
    cipher = AES.new(aes_key, AES.MODE_EAX)

    # Encrypt message
    ciphertext, tag = cipher.encrypt_and_digest(message.encode())

    return {
        "aes_key": aes_key,
        "ciphertext": base64.b64encode(ciphertext).decode(),
        "nonce": base64.b64encode(cipher.nonce).decode(),
        "tag": base64.b64encode(tag).decode()
    }


# ============================================================
# AES DECRYPTION FUNCTION
# ============================================================

def aes_decrypt(aes_key, ciphertext, nonce, tag):

    # Recreate AES cipher
    cipher = AES.new(
        aes_key,
        AES.MODE_EAX,
        nonce=base64.b64decode(nonce)
    )

    # Decrypt and verify
    decrypted_message = cipher.decrypt_and_verify(
        base64.b64decode(ciphertext),
        base64.b64decode(tag)
    )

    return decrypted_message.decode()


# ============================================================
# RSA ENCRYPT AES KEY
# ============================================================

def rsa_encrypt_key(aes_key, public_key):

    cipher_rsa = PKCS1_OAEP.new(public_key)

    encrypted_key = cipher_rsa.encrypt(aes_key)

    return base64.b64encode(encrypted_key).decode()


# ============================================================
# RSA DECRYPT AES KEY
# ============================================================

def rsa_decrypt_key(encrypted_key, private_key):

    cipher_rsa = PKCS1_OAEP.new(private_key)

    decrypted_key = cipher_rsa.decrypt(
        base64.b64decode(encrypted_key)
    )

    return decrypted_key


# ============================================================
# DIGITAL SIGNATURE FUNCTION
# ============================================================

def sign_message(message, private_key):

    # Create SHA256 hash
    hash_value = SHA256.new(message.encode())

    # Sign hash
    signature = pkcs1_15.new(private_key).sign(hash_value)

    return base64.b64encode(signature).decode()


# ============================================================
# SIGNATURE VERIFICATION FUNCTION
# ============================================================

def verify_signature(message, signature, public_key):

    hash_value = SHA256.new(message.encode())

    try:
        pkcs1_15.new(public_key).verify(
            hash_value,
            base64.b64decode(signature)
        )

        return True

    except (ValueError, TypeError):
        return False


# ============================================================
# USER INPUT
# ============================================================

email_message = input("\nEnter the e-mail message: ")


# ============================================================
# AES ENCRYPTION
# ============================================================

print("\nEncrypting message using AES...")

encrypted_data = aes_encrypt(email_message)


# ============================================================
# LOAD RECEIVER PUBLIC KEY
# ============================================================

receiver_public_key = load_public_key(
    "Keys/Receiver_Public.pem"
)


# ============================================================
# RSA ENCRYPT AES KEY
# ============================================================

encrypted_aes_key = rsa_encrypt_key(
    encrypted_data["aes_key"],
    receiver_public_key
)

print("AES key encrypted using RSA.")


# ============================================================
# LOAD SENDER PRIVATE KEY
# ============================================================

sender_private_key = load_private_key(
    "Keys/Sender_Private.pem"
)


# ============================================================
# CREATE DIGITAL SIGNATURE
# ============================================================

signature = sign_message(
    email_message,
    sender_private_key
)

print("Digital signature generated.")


# ============================================================
# CREATE SECURE EMAIL PACKET
# ============================================================

secure_email_packet = {
    "encrypted_message": encrypted_data["ciphertext"],
    "nonce": encrypted_data["nonce"],
    "tag": encrypted_data["tag"],
    "encrypted_aes_key": encrypted_aes_key,
    "signature": signature
}


# ============================================================
# SAVE PACKET AS JSON FILE
# ============================================================

with open("Messages/Secure_Email.json", "w") as f:
    json.dump(secure_email_packet, f, indent=4)

print("\nSecure e-mail packet created successfully.")


# ============================================================
# READ PACKET
# ============================================================

with open("messages/Secure_Email.json", "r") as f:
    received_packet = json.load(f)

print("Secure e-mail packet received.")


# ============================================================
# LOAD RECEIVER PRIVATE KEY
# ============================================================

receiver_private_key = load_private_key(
    "Keys/receiver_private.pem"
)


# ============================================================
# DECRYPT AES KEY
# ============================================================

decrypted_aes_key = rsa_decrypt_key(
    received_packet["encrypted_aes_key"],
    receiver_private_key
)

print("AES key decrypted.")


# ============================================================
# DECRYPT ORIGINAL MESSAGE
# ============================================================

decrypted_message = aes_decrypt(
    decrypted_aes_key,
    received_packet["encrypted_message"],
    received_packet["nonce"],
    received_packet["tag"]
)


# ============================================================
# DISPLAY DECRYPTED MESSAGE
# ============================================================

print("\nDecrypted Message:")
print(decrypted_message)


# ============================================================
# LOAD SENDER PUBLIC KEY
# ============================================================

sender_public_key = load_public_key(
    "Keys/Sender_Public.pem"
)


# ============================================================
# VERIFY DIGITAL SIGNATURE
# ============================================================

is_verified = verify_signature(
    decrypted_message,
    received_packet["signature"],
    sender_public_key
)


# ============================================================
# DISPLAY VERIFICATION RESULT
# ============================================================

print("\nSignature Verification Status:")

if is_verified:
    print("Signature Verified Successfully.")
else:
    print("Signature Verification Failed.")


# ============================================================
# COMPLETION MESSAGE
# ============================================================

print("\n" + "=" * 60)
print("SECURE COMMUNICATION COMPLETED")
print("=" * 60)

SECURE E-MAIL COMMUNICATION USING RSA AND AES
Folders created successfully.
Sender RSA keys generated.
Receiver RSA keys generated.

Encrypting message using AES...
AES key encrypted using RSA.
Digital signature generated.

Secure e-mail packet created successfully.
Secure e-mail packet received.
AES key decrypted.

Decrypted Message:
Hello this is a secure email

Signature Verification Status:
Signature Verified Successfully.

SECURE COMMUNICATION COMPLETED
